In [1]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()  # читає .env файл

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print("✅ Підключення успішне!")

✅ Підключення успішне!


In [2]:
RAW = r"C:\Users\Lubov\Downloads\olist_project\data\raw\\"

files = os.listdir(RAW)
print(f"Знайдено файлів: {len(files)}")
for f in files:
    print(" ", f)

Знайдено файлів: 9
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_orders_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv


In [3]:
from sqlalchemy import text
with engine.connect() as conn:
    tables = ['customers', 'orders', 'products', 'sellers', 
              'order_items', 'order_reviews', 'order_payments']
    
    for table in tables:
        result = conn.execute(text(f"DESCRIBE {table};"))
        rows = result.fetchall()
        print(f"\n📋 {table}:")
        for row in rows:
            print(f"   {row[0]:40} {row[1]}")


📋 customers:
   customer_id                              varchar(50)
   customer_unique_id                       varchar(50)
   customer_zip_code_prefix                 varchar(10)
   customer_city                            text
   customer_state                           text

📋 orders:
   order_id                                 varchar(50)
   customer_id                              varchar(50)
   order_status                             text
   order_purchase_timestamp                 datetime
   order_approved_at                        datetime
   order_delivered_carrier_date             datetime
   order_delivered_customer_date            datetime
   order_estimated_delivery_date            datetime

📋 products:
   product_id                               varchar(50)
   product_category_name                    varchar(100)
   product_name_lenght                      double
   product_description_lenght               double
   product_photos_qty                       double
   p

In [4]:
import pandas as pd
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
print(f"Всього рядків: {len(reviews)}")
print(f"Унікальних review_id: {reviews['review_id'].nunique()}")
print(f"Дублікатів: {len(reviews) - reviews['review_id'].nunique()}")

Всього рядків: 99224
Унікальних review_id: 98410
Дублікатів: 814


In [5]:
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")

# видаляємо дублікати по review_id
reviews.drop_duplicates(subset="review_id", keep="last", inplace=True)

# додаємо is_bad_review
reviews["is_bad_review"] = (reviews["review_score"] <= 2).astype(int)

# перезаливаємо таблицю
reviews.to_sql("order_reviews", engine, if_exists="replace", index=False, chunksize=5000)

print(f"✅ order_reviews: {len(reviews)} рядків після видалення дублікатів")

✅ order_reviews: 98410 рядків після видалення дублікатів


In [6]:
from sqlalchemy import text
import pandas as pd

with engine.connect() as conn:

    # ── крок 1: додаємо відсутні категорії ───────────────────────
    missing_cats = [
        ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_food_preparers"),
        ("pc_gamer", "pc_gamer"),
        ("unknown", "unknown"),
    ]
    for pt, en in missing_cats:
        conn.execute(text("""
            INSERT IGNORE INTO category_translation 
            (product_category_name, product_category_name_english)
            VALUES (:pt, :en)
        """), {"pt": pt, "en": en})
    conn.commit()
    print("✅ Відсутні категорії додано")

    # ── крок 2: NULL категорії → unknown ─────────────────────────
    conn.execute(text("""
        UPDATE products 
        SET product_category_name = 'unknown'
        WHERE product_category_name IS NULL;
    """))
    conn.commit()
    print("✅ NULL категорії замінено на 'unknown'")

    # ── крок 3: виправляємо ВСІ типи ─────────────────────────────
    all_fixes = [
        # customers
        "ALTER TABLE customers MODIFY customer_id VARCHAR(50) NOT NULL;",
        "ALTER TABLE customers MODIFY customer_unique_id VARCHAR(50);",
        "ALTER TABLE customers MODIFY customer_zip_code_prefix VARCHAR(10);",

        # orders
        "ALTER TABLE orders MODIFY order_id VARCHAR(50) NOT NULL;",
        "ALTER TABLE orders MODIFY customer_id VARCHAR(50);",
        "ALTER TABLE orders MODIFY order_purchase_timestamp DATETIME;",
        "ALTER TABLE orders MODIFY order_approved_at DATETIME;",
        "ALTER TABLE orders MODIFY order_delivered_carrier_date DATETIME;",
        "ALTER TABLE orders MODIFY order_delivered_customer_date DATETIME;",
        "ALTER TABLE orders MODIFY order_estimated_delivery_date DATETIME;",

        # products
        "ALTER TABLE products MODIFY product_id VARCHAR(50) NOT NULL;",
        "ALTER TABLE products MODIFY product_category_name VARCHAR(100);",

        # sellers
        "ALTER TABLE sellers MODIFY seller_id VARCHAR(50) NOT NULL;",
        "ALTER TABLE sellers MODIFY seller_zip_code_prefix VARCHAR(10);",

        # order_items
        "ALTER TABLE order_items MODIFY order_id VARCHAR(50);",
        "ALTER TABLE order_items MODIFY product_id VARCHAR(50);",
        "ALTER TABLE order_items MODIFY seller_id VARCHAR(50);",
        "ALTER TABLE order_items MODIFY shipping_limit_date DATETIME;",
        "ALTER TABLE order_items MODIFY price DECIMAL(10,2);",
        "ALTER TABLE order_items MODIFY freight_value DECIMAL(10,2);",

        # order_reviews
        "ALTER TABLE order_reviews MODIFY review_id VARCHAR(50) NOT NULL;",
        "ALTER TABLE order_reviews MODIFY order_id VARCHAR(50);",
        "ALTER TABLE order_reviews MODIFY review_score TINYINT;",
        "ALTER TABLE order_reviews MODIFY review_creation_date DATETIME;",
        "ALTER TABLE order_reviews MODIFY review_answer_timestamp DATETIME;",
        "ALTER TABLE order_reviews MODIFY is_bad_review TINYINT;",

        # order_payments
        "ALTER TABLE order_payments MODIFY order_id VARCHAR(50);",
        "ALTER TABLE order_payments MODIFY payment_value DECIMAL(10,2);",

        # category_translation
        "ALTER TABLE category_translation MODIFY product_category_name VARCHAR(100) NOT NULL;",
    ]
    for sql in all_fixes:
        conn.execute(text(sql))
    conn.commit()
    print("✅ Всі типи виправлено")

    # ── крок 4: видаляємо старі ключі ────────────────────────────
    drop_fk = [
        "ALTER TABLE orders DROP FOREIGN KEY fk_orders_customers;",
        "ALTER TABLE order_items DROP FOREIGN KEY fk_items_orders;",
        "ALTER TABLE order_items DROP FOREIGN KEY fk_items_products;",
        "ALTER TABLE order_items DROP FOREIGN KEY fk_items_sellers;",
        "ALTER TABLE order_reviews DROP FOREIGN KEY fk_reviews_orders;",
        "ALTER TABLE order_payments DROP FOREIGN KEY fk_payments_orders;",
        "ALTER TABLE products DROP FOREIGN KEY fk_products_category;",
    ]
    drop_pk = [
        "ALTER TABLE customers DROP PRIMARY KEY;",
        "ALTER TABLE orders DROP PRIMARY KEY;",
        "ALTER TABLE products DROP PRIMARY KEY;",
        "ALTER TABLE sellers DROP PRIMARY KEY;",
        "ALTER TABLE order_reviews DROP PRIMARY KEY;",
        "ALTER TABLE category_translation DROP PRIMARY KEY;",
    ]
    for sql in drop_fk + drop_pk:
        try:
            conn.execute(text(sql))
        except:
            pass
    conn.commit()
    print("✅ Старі ключі видалено")

    # ── крок 5: додаємо PRIMARY KEYS ─────────────────────────────
    pk_fixes = [
        "ALTER TABLE customers ADD PRIMARY KEY (customer_id);",
        "ALTER TABLE orders ADD PRIMARY KEY (order_id);",
        "ALTER TABLE products ADD PRIMARY KEY (product_id);",
        "ALTER TABLE sellers ADD PRIMARY KEY (seller_id);",
        "ALTER TABLE order_reviews ADD PRIMARY KEY (review_id);",
        "ALTER TABLE category_translation ADD PRIMARY KEY (product_category_name);",
    ]
    for sql in pk_fixes:
        conn.execute(text(sql))
    conn.commit()
    print("✅ Primary Keys додано")

    # ── крок 6: додаємо FOREIGN KEYS ─────────────────────────────
    fk_fixes = [
        """ALTER TABLE orders
           ADD CONSTRAINT fk_orders_customers
           FOREIGN KEY (customer_id) REFERENCES customers(customer_id);""",

        """ALTER TABLE order_items
           ADD CONSTRAINT fk_items_orders
           FOREIGN KEY (order_id) REFERENCES orders(order_id);""",

        """ALTER TABLE order_items
           ADD CONSTRAINT fk_items_products
           FOREIGN KEY (product_id) REFERENCES products(product_id);""",

        """ALTER TABLE order_items
           ADD CONSTRAINT fk_items_sellers
           FOREIGN KEY (seller_id) REFERENCES sellers(seller_id);""",

        """ALTER TABLE order_reviews
           ADD CONSTRAINT fk_reviews_orders
           FOREIGN KEY (order_id) REFERENCES orders(order_id);""",

        """ALTER TABLE order_payments
           ADD CONSTRAINT fk_payments_orders
           FOREIGN KEY (order_id) REFERENCES orders(order_id);""",

        """ALTER TABLE products
           ADD CONSTRAINT fk_products_category
           FOREIGN KEY (product_category_name)
           REFERENCES category_translation(product_category_name);""",
    ]
    for sql in fk_fixes:
        conn.execute(text(sql))
    conn.commit()
    print("✅ Foreign Keys додано")

print("\n🎉 Всі зв'язки між таблицями встановлено!")

✅ Відсутні категорії додано
✅ NULL категорії замінено на 'unknown'
✅ Всі типи виправлено
✅ Старі ключі видалено
✅ Primary Keys додано
✅ Foreign Keys додано

🎉 Всі зв'язки між таблицями встановлено!
